# ZeKamu — hakem düzeltmeleri için yeniden hesaplama (Colab)

Bu defter, hocanın istediği yeniden hesaplamaları **API çağrısı yapmadan** (mevcut `runs.jsonl` çıktıları üzerinden) ve
yalnızca iki yerde GPU/gömme gerektirerek çalıştırır.

| Bölüm | Ne yapar | Girdi | Süre |
|---|---|---|---|
| 1 | Çekimserlik ölçütü düzeltmesi; Tablo 2 ve Tablo 3'ün yeniden hesabı | `results/ans/runs.jsonl`, `results/ret/runs.jsonl` | saniyeler |
| 2a | Yıl göstergesi (R) çıkarılınca T ve sıralama | `results/ind.csv` | saniyeler |
| 2b | Sayfa numarası ve genel terim etkisi (H, A) | `data_kurum/*.pdf` | birkaç dakika |
| 3 | Model puanları gizli, aynı belge parçalarıyla kör insan değerlendirme dosyaları + κ | `index_bge/`, `goldset.csv` | birkaç dakika (bge-m3 yüklenir) |

**Çalıştırma sırası:** 0 → 1 → 2a → 2b → 3 (dosyaları üret) → *dosyaları elle puanla* → 3 (κ hücresi).
Tüm çıktılar `results/revision_v2/` altına yazılır.

## 0. Kurulum

In [ ]:
#@title Yollar
import os, sys, re, json, math, random, csv
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    get_ipython().system("pip -q install openpyxl scipy pymupdf sentence-transformers langchain-text-splitters")

BASE_STR = "/content/drive/MyDrive/ZeKamu"  #@param {type:"string"}
# Kod GitHub deposundan klonlandıysa buraya yolunu yaz (boş bırakılırsa Drive'daki kopyalar kullanılır)
CODE_DIR_STR = ""  #@param {type:"string"}

BASE = Path(os.environ.get("ZK_BASE") or BASE_STR)
CODE_DIR = Path(os.environ.get("ZK_CODE") or CODE_DIR_STR) if (os.environ.get("ZK_CODE") or CODE_DIR_STR) else None
for p in [BASE / "zekamu_eval", BASE] + ([CODE_DIR] if CODE_DIR else []):   # sonuncu eklenen önce aranır
    sys.path.insert(0, str(p))

OUT = BASE / "results" / "revision_v2"
OUT.mkdir(parents=True, exist_ok=True)
print("BASE:", BASE, "| mevcut:", BASE.exists())
print("OUT :", OUT)

## 1. Çekimserlik ölçütünün düzeltilmesi

**Sorun.** Önceki "esnek" tanım, yanıtta *herhangi bir yerde* "belirtilmemiştir / yer almamaktadır / bulunamadı" gibi bir ifade
geçiyorsa yanıtı çekimser sayıyordu. Böylece doğru bilgiyi veren ama aynı cümlede başka bir şeyin raporda olmadığını söyleyen
yanıtlar (örn. q009) "ret" olarak sayılıyor; hem gereksiz çekimserlik şişiyor hem de doğru yanıtlar düşüyordu.

**Düzeltme (altın cevaba bakmaz, yani döngüsel değildir).** Bir yanıt ancak
(i) sabit ret cümlesini içeriyorsa **veya** (ii) *tüm* cümleleri ret ifadesi taşıyorsa ve ret cümleleri çıkarıldığında geriye
içerik kalmıyorsa (< 3 anlamlı sözcük) çekimser sayılır. Kısmen yanıtlayıp bir kısmını reddeden yanıt **çekimser değildir**.

Hücre, eski tanımı da yeniden kurar ve makaledeki değerlerle karşılaştırır (kalibrasyon). Sapma varsa
`REFUSAL_RE` eski deftere göre ayarlanabilir; **düzeltilmiş tanımın mantığı değişmez.**

In [ ]:
import numpy as np, pandas as pd
from common import normalize_answer, wilson, mcnemar_exact, tr_lower

ABSTAIN = "Belgelerde bu bilgi bulunamadı."
CITE_RE = re.compile(r"\[Kaynak:[^\]]*\]")
# Ret ifadeleri (küçük harfe çevrilmiş metinde aranır)
REFUSAL_RE = re.compile(r"bulunamad|belirtilmem|yer almamakta|yer almamaktadır|yer almıyor|yer verilmem|mevcut değil|"
                        r"ifade edilmem|bilgi yok|verilmem|tespit edilemed|rastlanmad")

def body(a):            return CITE_RE.sub("", a or "").strip()
def sentences(t):       return [s for s in re.split(r"(?<=[.!?])\s+|\n+", t) if s.strip()]
def abstain_strict(a):  return normalize_answer(ABSTAIN) in normalize_answer(a or "")
def abstain_old(a):     return abstain_strict(a) or bool(REFUSAL_RE.search(tr_lower(a or "")))   # ESKİ esnek tanım (yeniden kurgu)

def abstain_fixed(a, min_tokens=3):
    """Yalnızca yanıtın TAMAMI ret ise çekimser."""
    if abstain_strict(a):
        return True
    sents = sentences(body(a))
    keep = [s for s in sents if not REFUSAL_RE.search(tr_lower(s))]
    if len(keep) == len(sents):          # hiç ret ifadesi yok
        return False
    return len(normalize_answer(" ".join(keep)).split()) < min_tokens

def contains_gold(a, g):
    ng = normalize_answer(g or "")
    return bool(ng) and ng in normalize_answer(body(a))

def tr(x, d=3):  return "-" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.{d}f}".replace(".", ",")
def frac(k, n, d=3):
    if n == 0: return "0/0"
    lo, hi = wilson(int(k), int(n))
    return f"{int(k)}/{int(n)} = {tr(k/n, d)} [{tr(lo, 2)}; {tr(hi, 2)}]"

ans = pd.read_json(BASE / "results/ans/runs.jsonl", lines=True)
ret = pd.read_json(BASE / "results/ret/runs.jsonl", lines=True)
ans["answerable"] = ans["answerable"].astype(bool)
ans["model_answer"] = ans["model_answer"].fillna("")
ans["abs_strict"] = ans.model_answer.map(abstain_strict)
ans["abs_old"]    = ans.model_answer.map(abstain_old)
ans["abs_fixed"]  = ans.model_answer.map(abstain_fixed)
ans["contains"]   = [contains_gold(a, g) if ok else False for a, g, ok in zip(ans.model_answer, ans.gold_answer, ans.answerable)]
CONDS = ["norag", "vector", "bm25", "hybrid"]
A_ = ans[ans.answerable]
print(f"{ans.id.nunique()} soru ({A_.id.nunique()} cevaplanabilir), koşullar: {sorted(ans.condition.unique())}")

# --- Kalibrasyon: yeniden kurulan eski tanım makaledeki değerleri veriyor mu? ---
PAPER = {"abs_strict": dict(norag=56, vector=13, bm25=6, hybrid=10),
         "abs_old":    dict(norag=65, vector=25, bm25=34, hybrid=32),
         "contains":   dict(norag=0, vector=16, bm25=15, hybrid=19)}
rows = []
for c in CONDS:
    d = A_[A_.condition == c]; s = d[d.qtype.isin(["kisa", "sayisal"])]
    rows.append(dict(kosul=c,
        strict_makale=PAPER["abs_strict"][c], strict_yeni=int(d.abs_strict.sum()),
        esnek_makale=PAPER["abs_old"][c],     esnek_yeniden_kurulan=int(d.abs_old.sum()),
        icerir_makale=PAPER["contains"][c],   icerir_yeni=int(s.contains.sum()),
        DUZELTILMIS_cekimser=int(d.abs_fixed.sum())))
calib = pd.DataFrame(rows); print(calib.to_string(index=False)); calib.to_csv(OUT / "calibration_abstention.csv", index=False)

In [ ]:
# --- Denetim dosyası: eski ve düzeltilmiş tanım ayrışan satırlar (ELLE KONTROL ET) ---
aud = ans[(ans.abs_old != ans.abs_fixed) & ans.answerable].copy()
aud["neden"] = np.where(aud.abs_old & ~aud.abs_fixed, "eski=çekimser, düzeltilmiş=cevap verdi", "eski=cevap, düzeltilmiş=çekimser")
cols = ["id", "condition", "qtype", "neden", "contains", "gold_answer", "model_answer"]
aud[cols].to_csv(OUT / "abstention_audit.csv", index=False, encoding="utf-8-sig")
print(f"{len(aud)} satır ayrışıyor -> abstention_audit.csv (hepsini gözle kontrol et)")
print(aud.groupby(["condition", "neden"]).size().to_string())

# Duyarlılık: min_tokens eşiği
for mt in (2, 3, 5):
    print(f"min_tokens={mt}:", {c: int(sum(abstain_fixed(a, mt) for a in A_[A_.condition == c].model_answer)) for c in CONDS})

### Tablo 2 ve Tablo 3 (düzeltilmiş) + metindeki sayılar

In [ ]:
def pick(cond): return A_[A_.condition == cond].set_index("id")
T2 = []
for c in CONDS:
    d = A_[A_.condition == c]; s = d[d.qtype.isin(["kisa", "sayisal"])]
    cite = d.cite_any_correct.astype(float) if c != "norag" else None
    T2.append({
        "Koşul": {"norag": "RAG'siz", "vector": "Vektör", "bm25": "BM25", "hybrid": "Hibrit"}[c],
        f"İçerir (n={len(s)})": frac(s.contains.sum(), len(s)),
        f"Doğru atıf (n={len(d)})": "–" if c == "norag" else frac(d.cite_any_correct.astype(bool).sum(), len(d)),
        "Gereksiz çekimserlik, DÜZELTİLMİŞ": frac(d.abs_fixed.sum(), len(d)),
        "(katı)": frac(d.abs_strict.sum(), len(d)),
        "(eski esnek)": frac(d.abs_old.sum(), len(d)),
        "F1 sayısal": tr(d[d.qtype == "sayisal"].f1.mean()),
        "F1 kısa":    tr(d[d.qtype == "kisa"].f1.mean()),
        "F1 uzun":    tr(d[d.qtype == "uzun"].f1.mean()),
    })
T2 = pd.DataFrame(T2); print(T2.to_string(index=False)); T2.to_csv(OUT / "tablo2_duzeltilmis.csv", index=False, encoding="utf-8-sig")

# Cevapsız sorularda ret (tüm tanımlar)
U_ = ans[~ans.answerable]
print("\nCevapsız sorularda çekimserlik:", {c: f"{int(U_[U_.condition==c].abs_fixed.sum())}/{int((U_.condition==c).sum())}" for c in CONDS})

# Soru türüne göre gereksiz çekimserlik (düzeltilmiş)
print("\nSoru türüne göre gereksiz çekimserlik (düzeltilmiş):")
print(A_.groupby(["condition", "qtype"]).abs_fixed.agg(["sum", "count"]).unstack(0).to_string())

In [ ]:
# --- Tablo 3: doğru sayfa ilk 5'te (isabet) / dışında (ıska) ---
ret["hit5"] = ret.first_hit_rank.notna() & (ret.first_hit_rank <= 5)
hit = ret.pivot_table(index="id", columns="method", values="hit5", aggfunc="first")
rows = []; T3_ex = []
for c in ["vector", "bm25", "hybrid"]:
    d = A_[A_.condition == c].copy()
    d["hit5"] = d.id.map(hit[c]).astype(bool)
    r = {"Koşul": {"vector": "Vektör", "bm25": "BM25", "hybrid": "Hibrit"}[c]}
    for lab, g in (("İsabet", d[d.hit5]), ("Iska", d[~d.hit5])):
        s = g[g.qtype.isin(["kisa", "sayisal"])]
        r[f"{lab} (n)"] = len(g)
        r[f"{lab} içerir"] = f"{int(s.contains.sum())}/{len(s)} = {tr(s.contains.mean())}" if len(s) else "0/0"
        r[f"{lab} çekimserlik (düz.)"] = f"{int(g.abs_fixed.sum())}/{len(g)} = {tr(g.abs_fixed.mean())}" if len(g) else "0/0"
        r[f"{lab} çekimserlik (eski)"] = f"{int(g.abs_old.sum())}/{len(g)}"
    rows.append(r)
    # çekimser OLMAYAN kısa/sayısal yanıtlarda doğruluk (seçilim etkisini görmek için)
    s = d[d.qtype.isin(["kisa", "sayisal"]) & ~d.abs_fixed]
    T3_ex.append((c, int(s.contains.sum()), len(s)))
T3 = pd.DataFrame(rows); print(T3.to_string(index=False)); T3.to_csv(OUT / "tablo3_duzeltilmis.csv", index=False, encoding="utf-8-sig")
print("\nÇekimser olmayan kısa/sayısal yanıtlarda içerir doğruluğu (düzeltilmiş tanım):")
for c, k, n in T3_ex: print(f"  {c:7s} {frac(k, n)}")

# Eşleştirilmiş karşılaştırma (içerir, kısa+sayısal)
S = {c: pick(c)[lambda x: x.qtype.isin(["kisa", "sayisal"])].contains for c in ["vector", "bm25", "hybrid"]}
idx = S["vector"].index
print("\nMcNemar (içerir):")
for a, b in (("vector", "hybrid"), ("vector", "bm25"), ("bm25", "hybrid")):
    print(f"  {a} vs {b}:", mcnemar_exact(list(S[a].loc[idx]), list(S[b].loc[idx])))

# Hibrit: Bölüm 4.6 sayıları
h = A_[A_.condition == "hybrid"].copy(); h["hit5"] = h.id.map(hit["hybrid"]).astype(bool)
print("\nHibrit, düzeltilmiş çekimserlik, soru türü:", h.groupby("qtype").abs_fixed.agg(["sum", "count"]).to_dict("index"))
print("Hibrit: çekimser sayılan ama doğru sayfayı atfeden:", int((h.abs_fixed & h.cite_any_correct.astype(bool)).sum()))
print("Hibrit: cevap veren yanıt sayısı (groundedness/insan değerlendirme havuzu):", int((~h.abs_fixed).sum()), "| eski tanımla:", int((~h.abs_old).sum()))

## 2. Bileşik puan: göstergeler ne ölçüyor?

**2a — Yıl göstergesi (R) çıkarılınca.** R yalnız rapor yılının doğrusal bir fonksiyonudur (2023/24/25 → 25/50/75). Bu hücre T'yi
R olmadan (A, C, H ağırlıkları 0,25/0,35/0,20 → toplam 0,80'e bölünerek) yeniden hesaplar ve yıllar arası farkın ve sıralamanın ne
kadarının R'den geldiğini gösterir. Aynı yıl içinde R sabit olduğundan, yıl içi sıralama T ve T(R'siz) için **özdeştir**.

In [ ]:
from scipy.stats import kendalltau, spearmanr, kruskal
ind = pd.read_csv(BASE / "results/ind.csv", encoding="utf-8-sig")
W = dict(A=.25, C=.35, R=.20, H=.20)
ind["T_check"] = sum(W[k] * ind[k] for k in "ACRH"); assert (ind["T_check"] - ind["T"]).abs().max() < 0.02, "ind.csv T ile uyuşmuyor"
ind["T_noR"] = (W["A"]*ind.A + W["C"]*ind.C + W["H"]*ind.H) / (W["A"] + W["C"] + W["H"])

def eta2(x, g):
    grand = x.mean(); ssb = sum(len(v) * (v.mean() - grand) ** 2 for _, v in x.groupby(g)); return ssb / ((x - grand) ** 2).sum()

def mc_tau(X, w0, delta=.2, n=10000, seed=42):
    rng = np.random.default_rng(seed); w0 = np.asarray(w0, float) / np.sum(w0)
    base = X @ w0; br = (-base).argsort().argsort() + 1; taus, shifts = [], []
    for _ in range(n):
        w = w0 * rng.uniform(1 - delta, 1 + delta, len(w0)); w /= w.sum(); s = X @ w
        taus.append(kendalltau(base, s).statistic); shifts.append(np.abs((-s).argsort().argsort() + 1 - br).mean())
    return dict(tau_mean=float(np.mean(taus)), tau_p5=float(np.quantile(taus, .05)), tau_min=float(np.min(taus)), mean_abs_rank_shift=float(np.mean(shifts)))

yr = ind.groupby("year")[["A", "C", "H", "R", "T", "T_noR"]].mean().round(2)
print(yr.to_string())
gap = lambda c: ind.groupby("year")[c].mean()
res = {
  "yil_farki_2025_2023_T": float(gap("T")[2025] - gap("T")[2023]),
  "yil_farki_2025_2023_T_Rsiz": float(gap("T_noR")[2025] - gap("T_noR")[2023]),
  "eta2_yil_T": float(eta2(ind["T"], ind.year)), "eta2_yil_T_Rsiz": float(eta2(ind["T_noR"], ind.year)),
  "kruskal_p_T": float(kruskal(*[g["T"] for _, g in ind.groupby("year")]).pvalue),
  "kruskal_p_T_Rsiz": float(kruskal(*[g["T_noR"] for _, g in ind.groupby("year")]).pvalue),
  "kendall_tau_T_vs_T_Rsiz": float(kendalltau(ind["T"], ind["T_noR"]).statistic),
  "spearman_T_vs_T_Rsiz": float(spearmanr(ind["T"], ind["T_noR"]).statistic),
}
res["mc_T_agirlik_%20"]      = mc_tau(ind[list("ACRH")].values, [W[k] for k in "ACRH"])
res["mc_T_Rsiz_agirlik_%20"] = mc_tau(ind[list("ACH")].values,  [W[k] for k in "ACH"])
print(json.dumps(res, ensure_ascii=False, indent=2))
(OUT / "yil_gostergesi_cikarma.json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
yr.to_csv(OUT / "yil_ortalamalari.csv")

**2b — Sayfa numarası ve genel terimler.** `H`, "bulgu/sorumluluk terimi **ve** sayı içeren sayfa oranı"dır. Sayfa altbilgisindeki
sayfa numarası, madde numaraları ve kanun numaraları da "sayı" sayıldığı için; ayrıca `yönetim`, `görüş`, `öneri` gibi terimler
her denetim sayfasında geçtiği için H'nin hesap verebilirliği doğrudan ölçmediği, yalnız bir vekil olduğu görülür. Bu hücre:

1. sayfa-numarası satırları çıkarılınca H ve A'nın nasıl değiştiğini,
2. yalnızca tutar benzeri sayılar (`1.234.567,89`, `… TL`, `%`) sayıldığında H'yi,
3. her terimin sayfaların yüzde kaçında geçtiğini (özellikle `yönetim`),
4. bu varyantlarla T'nin ve sıralamanın değişimini

hesaplar. PDF'leri `data_kurum/` içinden okur.

In [ ]:
from common import read_pdf_pages
PDF_DIR = BASE / "data_kurum"
TARGET  = ["hedef", "performans programı", "stratejik plan", "amaç"]
RESULT  = ["gerçekleş", "sonuç", "ulaşıl", "tamamlan"]
ACCOUNT = ["bulgu", "sorumlu", "hesap verebilir", "tavsiye", "öneri", "kamu zararı", "görüş", "yönetim"]
NUM  = re.compile(r"\d[\d.,]*")
AMT  = re.compile(r"\d{1,3}(?:\.\d{3})+(?:,\d+)?|\b\d+,\d{2}\b|\b\d+\s*(?:tl|₺|lira)\b|%\s*\d+|\b\d+\s*%|yüzde\s*\d+")
PAGE_LINE = re.compile(r"^\s*[-–—]?\s*(?:sayfa\s*)?\d{1,3}\s*(?:/\s*\d{1,3})?\s*[-–—]?\s*$", re.I)

def strip_page_numbers(t):
    return "\n".join(l for l in t.splitlines() if not PAGE_LINE.match(l))

def indicator_variants(pages):
    """Bir raporun sayfa metinlerinden A/C/H varyantları + terim kapsamı."""
    n = max(len(pages), 1)
    low = [tr_lower(p) for p in pages]
    low_np = [tr_lower(strip_page_numbers(p)) for p in pages]
    acc_hit = [any(a in p for a in ACCOUNT) for p in low]
    acc_hit_nomgmt = [any(a in p for a in ACCOUNT if a != "yönetim") for p in low]
    tgt = [p for p in low if any(t in p for t in TARGET)]
    out = dict(
        pages=len(pages),
        A=100 * sum(len(p) >= 200 for p in pages) / n,
        A_np=100 * sum(len(strip_page_numbers(p)) >= 200 for p in pages) / n,
        C=100 * sum(any(r in p for r in RESULT) for p in tgt) / len(tgt) if tgt else 0.0,
        H=100 * sum(a and bool(NUM.search(p)) for a, p in zip(acc_hit, low)) / n,
        H_np=100 * sum(a and bool(NUM.search(p)) for a, p in zip(acc_hit, low_np)) / n,
        H_amt=100 * sum(a and bool(AMT.search(p)) for a, p in zip(acc_hit, low_np)) / n,
        H_np_nomgmt=100 * sum(a and bool(NUM.search(p)) for a, p in zip(acc_hit_nomgmt, low_np)) / n,
        H_amt_nomgmt=100 * sum(a and bool(AMT.search(p)) for a, p in zip(acc_hit_nomgmt, low_np)) / n,
        # yalnızca sayfa numarası yüzünden H=1 olan sayfa sayısı
        pn_only_pages=sum(a and bool(NUM.search(p)) and not bool(NUM.search(q)) for a, p, q in zip(acc_hit, low, low_np)),
        h_pages=sum(a and bool(NUM.search(p)) for a, p in zip(acc_hit, low)),
    )
    for t in ACCOUNT:
        out[f"term_{t}"] = sum(t in p for p in low)
    return out

def run_variants():
    rows = []
    for pdf in sorted(PDF_DIR.rglob("*.pdf")):
        v = indicator_variants(read_pdf_pages(pdf)); v["doc"] = pdf.name; rows.append(v)
    return pd.DataFrame(rows)

if PDF_DIR.exists() and any(PDF_DIR.rglob("*.pdf")):
    V = run_variants()
else:
    print("UYARI: data_kurum içinde PDF bulunamadı; 2b atlandı"); V = None

In [ ]:
if V is not None:
    D = ind[["doc", "year", "A", "C", "R", "H", "T"]].merge(V, on="doc", suffixes=("", "_yeni"))
    print(f"{len(D)}/{len(ind)} rapor eşleşti")
    print("Baz çizgi kontrolü (ind.csv ile yeniden hesap farkı): max|A|=%.2f max|H|=%.2f max|C|=%.2f" %
          ((D.A - D.A_yeni).abs().max(), (D.H - D.H_yeni).abs().max(), (D.C - D.C_yeni).abs().max()))
    tot_pages = D.pages.sum()
    print(f"\nSayfa numarası etkisi: H'ye katkı veren {int(D.h_pages.sum())} sayfanın {int(D.pn_only_pages.sum())}'i "
          f"({100*D.pn_only_pages.sum()/D.h_pages.sum():.1f}%) yalnız sayfa numarası yüzünden 'sayı içeriyor' sayıldı.")
    print("\nOrtalama göstergeler:")
    print(D[["A", "A_np", "H", "H_np", "H_amt", "H_np_nomgmt", "H_amt_nomgmt"]].mean().round(2).to_string())

    # Terim kapsamı (sayfa yüzdesi)
    tc = {t: 100 * D[f"term_{t}"].sum() / tot_pages for t in ACCOUNT}
    print("\nH terimlerinin geçtiği sayfa oranı (%):"); print(pd.Series(tc).round(1).sort_values(ascending=False).to_string())

    # T varyantları ve sıralama
    for nm, h in (("T_np", "H_np"), ("T_amt", "H_amt"), ("T_amt_nomgmt", "H_amt_nomgmt")):
        D[nm] = W["A"] * D.A_np + W["C"] * D.C + W["R"] * D.R + W["H"] * D[h]
    D["T_noR"] = (W["A"] * D.A + W["C"] * D.C + W["H"] * D.H) / .8
    D["T_noR_amt"] = (W["A"] * D.A_np + W["C"] * D.C + W["H"] * D.H_amt) / .8
    rows = []
    for nm in ("T_np", "T_amt", "T_amt_nomgmt", "T_noR", "T_noR_amt"):
        g = D.groupby("year")[nm].mean()
        rows.append(dict(varyant=nm, ort=D[nm].mean(), yil2023=g[2023], yil2024=g[2024], yil2025=g[2025],
                         fark_2025_2023=g[2025] - g[2023], kendall_tau_T_ile=kendalltau(D["T"], D[nm]).statistic,
                         spearman_T_ile=spearmanr(D["T"], D[nm]).statistic, kruskal_p=kruskal(*[x[nm] for _, x in D.groupby("year")]).pvalue))
    rows.insert(0, dict(varyant="T (baz)", ort=D["T"].mean(), yil2023=D.groupby("year")["T"].mean()[2023], yil2024=D.groupby("year")["T"].mean()[2024],
                        yil2025=D.groupby("year")["T"].mean()[2025], fark_2025_2023=D.groupby("year")["T"].mean()[2025]-D.groupby("year")["T"].mean()[2023],
                        kendall_tau_T_ile=1.0, spearman_T_ile=1.0, kruskal_p=kruskal(*[x["T"] for _, x in D.groupby("year")]).pvalue))
    ABL = pd.DataFrame(rows).round(4); print("\n", ABL.to_string(index=False))
    ABL.to_csv(OUT / "indicator_ablation_summary.csv", index=False, encoding="utf-8-sig")
    D.to_csv(OUT / "indicator_ablation_per_report.csv", index=False, encoding="utf-8-sig")
    pd.Series(tc, name="sayfa_orani_%").to_csv(OUT / "h_terim_kapsami.csv")

## 3. Kör (model puanı gizli) insan değerlendirmesi — aynı belge parçaları üzerinden

Eski düzenin iki sorunu vardı: insan, hakem puanı görünür halde puanlamıştı; ve insan *kaynak sayfayı*, hakem ise *üretime verilen
beş bölütü* görmüştü. Burada her iki puanlayıcı da **modele verilen beş bölütün birebir aynısını** görür ve hakem puanı dosyada yoktur.

* Bağlamın birebir aynı olduğu, `llm_cache.jsonl` içindeki istem özetiyle (SHA) **doğrulanır**.
* Havuz: hibrit koşulu, cevaplanabilir, **düzeltilmiş tanıma göre çekimser olmayan** yanıtların tümü.
* İki bağımsız puanlayıcı için (A = sen, B = başka biri) ayrı karışık sıralı dosyalar üretilir; eşleme anahtarı ayrı dosyadadır
  (`blind_key.csv`) — puanlayıcılara **verme**.
* Dayanaklılık sayfasında altın cevap **yoktur**; doğruluk ayrı sayfadadır (altın cevapla) ve ayrı sırayla karıştırılmıştır.

In [ ]:
from common import Retriever, load_gold, sha
from eval_answers import PROMPT_RAG, build_context
MODEL, EFFORT = "openai/gpt-oss-120b", "low"      # eval_answers.py varsayılanları; farklıysa düzelt

gold = {g["id"]: g for g in load_gold(BASE / "goldset.csv", "test") if g["answerable"]}
pool = ans[(ans.condition == "hybrid") & ans.answerable & ~ans.abs_fixed].copy()
print("Havuz (hibrit, cevap veren):", len(pool))

r = Retriever(BASE / "index_bge")
cache_keys = set()
cp = BASE / "results/ans/llm_cache.jsonl"
if cp.exists():
    cache_keys = {json.loads(l)["k"] for l in cp.read_text(encoding="utf-8").splitlines() if l.strip()}

items, ok_cache = [], 0
for _, row in pool.iterrows():
    q = gold[row.id]["question"]
    ctx = build_context(r, r.search(q, "hybrid", 5))
    key = sha(f"{MODEL}\n{EFFORT}\n" + PROMPT_RAG.format(abstain=ABSTAIN, context=ctx, question=q))
    hit_ = key in cache_keys; ok_cache += hit_
    items.append(dict(id=row.id, soru=q, baglam=ctx, yanit=row.model_answer, altin=row.gold_answer, judge=row.grounded, cache_ok=hit_))
print(f"Bağlam doğrulaması: {ok_cache}/{len(items)} istem önbellekteki istemle BİREBİR eşleşti")
if cache_keys and ok_cache < len(items):
    print("UYARI: eşleşmeyen satırlar var; MODEL/EFFORT değerlerini ve indeks sürümünü kontrol et (aşağıdaki dosyalar yine de yazılır).")

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill

RUBRIK = ["PUANLAMA YÖNERGESİ (dayanaklılık)",
          "Yalnızca yanıtı ve 'Belge parçaları' sütunundaki metni kullan. Kendi bilgini ve başka kaynakları kullanma.",
          "2 = yanıttaki TÜM iddialar belge parçalarıyla destekleniyor",
          "1 = bazı iddialar destekleniyor, bazıları desteklenmiyor",
          "0 = iddialar desteklenmiyor veya parçalarla çelişiyor",
          "Atıf ([Kaynak: ...]) sayfasının doğruluğunu PUANLAMA; yalnızca iddiaların parçalarda bulunup bulunmadığına bak.",
          "",
          "PUANLAMA YÖNERGESİ (doğruluk) — ikinci sayfa",
          "2 = altın cevabın anlamını tam karşılıyor | 1 = kısmen karşılıyor | 0 = karşılamıyor / yanlış"]

def write_rater(label, seed):
    rng = random.Random(seed)
    o1 = list(range(len(items))); rng.shuffle(o1)
    o2 = list(range(len(items))); rng.shuffle(o2)
    wb = Workbook(); ws = wb.active; ws.title = "yonerge"
    for i, t in enumerate(RUBRIK, 1): ws.cell(i, 1, t)
    ws.column_dimensions["A"].width = 120
    w1 = wb.create_sheet("dayanaklilik"); w1.append(["sira", "soru", "belge_parcalari", "yanit", "insan_destek_0_1_2", "not"])
    for n, i in enumerate(o1, 1):
        it = items[i]; w1.append([n, it["soru"], it["baglam"], it["yanit"], None, None])
    w2 = wb.create_sheet("dogruluk"); w2.append(["sira", "soru", "altin_cevap", "yanit", "insan_dogru_0_1_2", "not"])
    for n, i in enumerate(o2, 1):
        it = items[i]; w2.append([n, it["soru"], it["altin"], it["yanit"], None, None])
    for w, widths in ((w1, (6, 45, 110, 55, 18, 30)), (w2, (6, 45, 45, 55, 18, 30))):
        for c, wd in zip("ABCDEF", widths): w.column_dimensions[c].width = wd
        for row in w.iter_rows(min_row=2):
            for c in row: c.alignment = Alignment(wrap_text=True, vertical="top")
        for c in w[1]: c.font = Font(bold=True); c.fill = PatternFill("solid", fgColor="DDEBF7")
        w.freeze_panes = "A2"
    wb.save(OUT / f"human_blind_rater{label}.xlsx")
    return [dict(rater=label, sheet="dayanaklilik", sira=n, id=items[i]["id"]) for n, i in enumerate(o1, 1)] + \
           [dict(rater=label, sheet="dogruluk", sira=n, id=items[i]["id"]) for n, i in enumerate(o2, 1)]

key = write_rater("A", 101) + write_rater("B", 202)
pd.DataFrame(key).to_csv(OUT / "blind_key.csv", index=False)
pd.DataFrame([{k: v for k, v in it.items() if k in ("id", "judge")} for it in items]).to_csv(OUT / "judge_scores_DO_NOT_SHARE.csv", index=False)
print(f"{len(items)} satırlık iki dosya yazıldı: human_blind_raterA.xlsx / human_blind_raterB.xlsx")
print("Puanlayıcılara YALNIZ bu iki xlsx'ten ilgili olanı ver; blind_key.csv ve judge_scores_DO_NOT_SHARE.csv'yi verme.")

### Puanlama bittikten sonra: κ
`human_blind_raterA.xlsx` (ve varsa `...B.xlsx`) dosyalarını doldurup aynı klasöre geri yükle, sonra hücreyi çalıştır.

In [ ]:
def wkappa(a, b, k=3):
    a, b = np.asarray(a, int), np.asarray(b, int)
    O = np.zeros((k, k))
    for x, y in zip(a, b): O[x, y] += 1
    E = np.outer(O.sum(1), O.sum(0)) / O.sum()
    W_ = np.array([[(i - j) ** 2 / (k - 1) ** 2 for j in range(k)] for i in range(k)])
    den = (W_ * E).sum()
    return 1 - (W_ * O).sum() / den if den else float("nan")

def boot_ci(a, b, n=2000, seed=42):
    rng = np.random.default_rng(seed); a, b = np.asarray(a), np.asarray(b); v = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a)); v.append(wkappa(a[i], b[i]))
    return tuple(np.nanquantile(v, [.025, .975]))

keydf = pd.read_csv(OUT / "blind_key.csv"); judge = pd.read_csv(OUT / "judge_scores_DO_NOT_SHARE.csv").set_index("id")["judge"]
def load_rater(label, sheet, col):
    p = OUT / f"human_blind_rater{label}.xlsx"
    if not p.exists(): return None
    d = pd.read_excel(p, sheet_name=sheet)
    d = d.dropna(subset=[col]).merge(keydf[(keydf.rater == label) & (keydf.sheet == sheet)], on="sira")
    return d.set_index("id")[col].astype(int)

report = {}
H = {L: load_rater(L, "dayanaklilik", "insan_destek_0_1_2") for L in "AB"}
for L, h in H.items():
    if h is None or len(h) == 0: continue
    j = (judge.loc[h.index] * 2).round().astype(int)
    lo, hi = boot_ci(j.values, h.values)
    report[f"hakem_vs_puanlayici{L}"] = dict(n=len(h), kappa_w=wkappa(j, h), ci95=[lo, hi], tam_uyum=float((j.values == h.values).mean()),
                                              ort_hakem=float(j.mean() / 2), ort_insan=float(h.mean() / 2),
                                              capraz=pd.crosstab(j, h).to_dict())
if all(H[L] is not None and len(H[L]) for L in "AB"):
    common_ids = H["A"].index.intersection(H["B"].index)
    lo, hi = boot_ci(H["A"].loc[common_ids].values, H["B"].loc[common_ids].values)
    report["puanlayiciA_vs_B"] = dict(n=len(common_ids), kappa_w=wkappa(H["A"].loc[common_ids], H["B"].loc[common_ids]), ci95=[lo, hi],
                                      tam_uyum=float((H["A"].loc[common_ids].values == H["B"].loc[common_ids].values).mean()))
for L in "AB":
    c = load_rater(L, "dogruluk", "insan_dogru_0_1_2")
    if c is not None and len(c): report[f"dogruluk_puanlayici{L}"] = dict(n=len(c), ortalama=float(c.mean() / 2), dagilim=c.value_counts().sort_index().to_dict())
print(json.dumps(report, ensure_ascii=False, indent=2, default=str) if report else "Henüz doldurulmuş dosya yok.")
if report: (OUT / "human_kappa_blind.json").write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

## 4. Çıktılar (repoya konacak)
`results/revision_v2/` altında: `calibration_abstention.csv`, `abstention_audit.csv`, `tablo2_duzeltilmis.csv`, `tablo3_duzeltilmis.csv`,
`yil_gostergesi_cikarma.json`, `yil_ortalamalari.csv`, `indicator_ablation_summary.csv`, `indicator_ablation_per_report.csv`,
`h_terim_kapsami.csv`, `human_blind_rater{A,B}.xlsx`, `human_kappa_blind.json`.

> `blind_key.csv` ve `judge_scores_DO_NOT_SHARE.csv` puanlama bitene kadar puanlayıcılarla paylaşılmamalı; puanlama bittikten sonra repoya konabilir.

In [ ]:
for p in sorted(OUT.glob("*")): print(f"{p.stat().st_size:>9,}  {p.name}")